# Tiny RAG

Use the TinyLama model to build a simple RAG system to answer a question about an event that was outside of the model's knowledge cutoff.


---
**IMPORTANT**

This notebook relies on their being a pdf file uploaded to `./data/2024-25_NFL_playoffs_pg_1.pdf`

Create the `./data` folder and upload [this](https://github.com/didiergarcia/tiny-rag/blob/main/data/2024%E2%80%9325_NFL_playoffs_pg_1.pdf
 ) file.

---

## Model Info
model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
format: int4 (4-bit quantized)
size on disk: ~700MB
ram used: ~1GB

TinyLlama-1.1B-Chat-v1.0 is a compact, 1.1 billion–parameter language model trained on 3 trillion tokens and fine-tuned for conversational use with chat-style alignment. Built to mirror the LLaMA architecture, it delivers strong performance for tasks like RAG and assistant-style Q&A while remaining efficient enough for edge devices and Colab environments.


## Vector DB
FAISS
http://github.com/facebookresearch/faiss


FAISS (Facebook AI Similarity Search) is an open-source library for fast, efficient vector similarity search and clustering of dense embeddings. It’s commonly used in RAG systems to retrieve relevant documents by comparing query vectors against millions of stored embeddings using various indexing strategies.


## Embeddings
all-MiniLM-L6-v2 sentence transformer

all-MiniLM-L6-v2 is a lightweight sentence transformer model that maps text to 384-dimensional embeddings optimized for semantic similarity tasks like clustering, retrieval, and ranking. Trained using a distillation approach from larger transformer models, it offers an excellent balance of speed and accuracy for real-time applications.

## Document Processing
PyMuPDF for PDF text extraction

PyMuPDF is a Python library that provides fast, lightweight access to PDF and other document formats, enabling text extraction, page manipulation, and metadata handling. Built on the MuPDF engine, it supports precise layout-aware text extraction, making it ideal for preprocessing PDFs in NLP and document analysis workflows.

## Install requirements

In [1]:
!pip install transformers sentence_transformers faiss-cpu bitsandbytes PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## Imports

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import faiss
import fitz
from sentence_transformers import SentenceTransformer

## Model

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Test Pre-RAG
Let's see what the model knows about our topic before we give access to new information via RAG.

We will ask the model which team won the 2024 NFL Championship.

Correct answer: The Philadelphia Eagles won the 2024 NFL Championship on February 9, 2025

But the model we're using has a knowledge cut-off for this model is somewhere in late 2023 so there is no chance it was trained on this information.

In [4]:
query = "Who won the 2024 NFL Championship?"
prompt = (
    f"<|system|>You are a help assistant.<|end|>\n"
    f"<|user|>Question: {query}<|end|><|assistant|>"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=100)
decoded_output = tokenizer.decode(out[0], skip_special_tokens=True)
print(decoded_output.split("<|assistant|>")[1])

Answer: The 2024 NFL Championship will be won by the Los Angeles Rams. The Los Angeles Rams are the defending champions of the 2024 NFL Championship.


### Result? A Hallucination
It looks like the model said the Los Angles Rams won the 2024 NFL Championship, but we know that it was actually the Philadelphia Eagles that won the championship.

Link: https://en.wikipedia.org/wiki/2024%E2%80%9325_NFL_playoffs


Let's add some "Context" for the model to fix this issue!

In [5]:
context = """The NFL playoffs for the 2024 season began on January 11, 2025, and concluded with Super Bowl LIX on February 9 at Caesars Superdome in New Orleans, Louisiana, when the Philadelphia Eagles defeated the defending champion Kansas City Chiefs 40–22 to win their second Super Bowl in franchise history."""

### NOTE
As you can see the "context" is just a string with some information from wikipedia about the 2024 Super Bowl Winner.

We cut a few corners here...

Normally there is a vector database that stores information that is used for context. That knowledge comes from text, PDFs, and other documents and usually has to be processed, chunked, and inserted into a Vector DB.

Instead we're using just this string and we're going to add it to our prompt as context to show the affect of setting up an actual Vector DB to our query.

In [7]:
query = "Who won the 2024 NFL Championship?"
prompt = (
    f"<|system|>You are a help assistant.<|end|>\n"
    f"<|user|>Context: {context} Question: {query}<|end|><|assistant|>"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=300)
decoded_output = tokenizer.decode(out[0], skip_special_tokens=True)
print(decoded_output.split("<|assistant|>")[1])


The 2024 NFL Championship was won by the Philadelphia Eagles, who defeated the defending champion Kansas City Chiefs 40-22 in the Super Bowl LIX at Caesars Superdome in New Orleans, Louisiana.


As you can see the model now returns the correct information thanks to the added context.

Let's add an actual Vector DB and setup a proper RAG system.

## RAG Setup
We're going to use FAISS as our Vector DB and we're going to add the contents of a first page of the PDF print out of the wikipedia arcticle about the 2024-2025 NFL playoffs which contains the information we need.

### Extract text from the PDF
We need to get at the text in the PDF. We'll use "fitz" to extract the text.

In [8]:
pdf_text = "\n".join(block[4] for block in fitz.open("./data/2024–25_NFL_playoffs_pg_1.pdf").load_page(0).get_text("blocks"))

In [9]:
pdf_text

"2024–25 NFL playoffs\n\nDates\nJanuary 11 – February 9,\n2025\n\nSeason\n2024\n\nTeams\n14\n\nGames played\n13\n\nSuper Bowl LIX\nsite\n\nCaesars Superdome\n\nNew Orleans, Louisiana\n\nDefending\nchampions\n\nKansas City Chiefs\n\nChampion\nPhiladelphia Eagles\n\nRunner-up\nKansas City Chiefs\n\nConference\nrunners-up\n\nBuffalo Bills\n\nWashington\nCommanders\n\nNFL playoffs\n← 2023–24\n\n2024–25 NFL playoffs\n\nThe NFL playoffs for the 2024 season began on January 11, 2025, and\nconcluded with Super Bowl LIX on February 9 at Caesars Superdome in\nNew Orleans, Louisiana, when the Philadelphia Eagles defeated the\ndefending champion Kansas City Chiefs 40–22 to win their second Super\nBowl in franchise history.\n\nAll playoff teams in both conferences won at least 10 games, the first time\nsince 2012 and the first time since the NFL playoffs expanded to a 14-\nteam format in 2020. This was the second time since their respective\nmoves to Los Angeles where the Rams and Chargers both qua

### Chunk the Text
We need to break up the text into smaller chunks to help find specific bits of information we're looking for and to fit within the embeddings length for the Vector DB.

In [10]:
chunk_size = 150
pdf_word_chunks = pdf_text.split() # Split on whitespace
pdf_chunks = [" ".join(pdf_word_chunks[i:i+chunk_size]) for i in range(0, len(pdf_word_chunks), chunk_size)]

print(f"{len(pdf_chunks)} chunks")

4 chunks


In [11]:
pdf_chunks

['2024–25 NFL playoffs Dates January 11 – February 9, 2025 Season 2024 Teams 14 Games played 13 Super Bowl LIX site Caesars Superdome New Orleans, Louisiana Defending champions Kansas City Chiefs Champion Philadelphia Eagles Runner-up Kansas City Chiefs Conference runners-up Buffalo Bills Washington Commanders NFL playoffs ← 2023–24 2024–25 NFL playoffs The NFL playoffs for the 2024 season began on January 11, 2025, and concluded with Super Bowl LIX on February 9 at Caesars Superdome in New Orleans, Louisiana, when the Philadelphia Eagles defeated the defending champion Kansas City Chiefs 40–22 to win their second Super Bowl in franchise history. All playoff teams in both conferences won at least 10 games, the first time since 2012 and the first time since the NFL playoffs expanded to a 14- team format in 2020. This was the second time since their respective moves to Los Angeles where the Rams and Chargers both',
 'qualified for the playoffs, after 2018. Within each conference, the fou

### Create Embeddings
We create Embeddings for each chunk. Embeddings are used to estimate the meaning of the chunk. When two embeddings are close to each other in geometrically in multi-dimensional space we know the chunks are close in meaning.

In [12]:
sentenceTransformerModel = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentenceTransformerModel.encode(pdf_chunks, convert_to_tensor=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Insert into Vector DB
Let's add our embeddings to the Vector DB so we can later query it.

In [13]:
dimension = embeddings[0].shape[0]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

### Test the Vector DB
Let's see if asking about the Championship brings up the right chunk of text.

In [14]:
query = "Who won the 2024 NFL Championship?"
q_embedding = sentenceTransformerModel.encode([query])
D, I = index.search(q_embedding, k=3)

for idx in I[0]:
    print("Match:", pdf_chunks[idx])

Match: 2024–25 NFL playoffs Dates January 11 – February 9, 2025 Season 2024 Teams 14 Games played 13 Super Bowl LIX site Caesars Superdome New Orleans, Louisiana Defending champions Kansas City Chiefs Champion Philadelphia Eagles Runner-up Kansas City Chiefs Conference runners-up Buffalo Bills Washington Commanders NFL playoffs ← 2023–24 2024–25 NFL playoffs The NFL playoffs for the 2024 season began on January 11, 2025, and concluded with Super Bowl LIX on February 9 at Caesars Superdome in New Orleans, Louisiana, when the Philadelphia Eagles defeated the defending champion Kansas City Chiefs 40–22 to win their second Super Bowl in franchise history. All playoff teams in both conferences won at least 10 games, the first time since 2012 and the first time since the NFL playoffs expanded to a 14- team format in 2020. This was the second time since their respective moves to Los Angeles where the Rams and Chargers both
Match: other two surviving teams play each other, with the higher-seed

As expected the first match is the chunk containing the information we need for the context!

## Query with RAG Context
We've now got a Vector DB setup to do our Retrival and we can try our initial query again.

In [15]:
query = "Who won the 2024 NFL Championship?"
q_embedding = sentenceTransformerModel.encode([query])
D, I = index.search(q_embedding, k=3)
context = pdf_chunks[I[0][0]]

prompt = (
    f"<|system|>You are a help assistant.<|end|>\n"
    f"<|user|>Context: {context} Question: {query}<|end|><|assistant|>"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=300)
decoded_output = tokenizer.decode(out[0], skip_special_tokens=True)
print(decoded_output.split("<|assistant|>")[1])

The 2024 NFL Championship was won by the Philadelphia Eagles, defeating the defending champion Kansas City Chiefs 40-22 in Super Bowl LIX at Caesars Superdome in New Orleans, Louisiana.


**Success!** The model is now aware of data from outside of it's original training data and can now successfully answer the question!
